# Number & Brightness Analysis Pipeline

## Overview
This notebook performs Number & Brightness (N&B) analysis on fluorescence microscopy time series data. N&B analysis is a technique used to determine the molecular brightness and apparent number of fluorescent molecules in each pixel by analyzing intensity fluctuations over time.

## Input/Output
- **Input**: 
  - `*.ptu` files (raw microscopy data)
  - Various polarization channels: `*_green_s.tif`, `*_green_p.tif`, `*_red_s_prompt.tif`, `*_red_p_prompt.tif`, `*_red_s_delay.tif`, `*_red_p_delay.tif`
  - ROI masks: `*_ROI*.tif` (from segmentation step)
- **Output**: 
  - Statistical parameter images: `*_mean_red.tif`, `*_variance_red.tif`, `*_brightness_red.tif`, `*_appNumber_red.tif`
  - Summary data table: `Number&Brightness_{channel}.txt`

## Configuration Parameters
- **Channel selection**: Currently configured for red delay channel (acceptor only samples)
- **Analysis type**: Frame-to-frame temporal analysis
- **Statistical measures**: Mean, variance, brightness (var/mean), apparent number (mean²/var)

## Mathematical Background
- **Brightness** = Variance/Mean (molecular brightness per particle)
- **Apparent Number** = Mean²/Variance (number of particles per pixel)
- These parameters are derived from photon counting statistics and provide information about molecular aggregation and concentration

## Processing Steps
1. Load time series data from multiple polarization channels
2. Combine channels according to sample type (donor-only, acceptor-only, or FRET)
3. Calculate temporal mean and variance for each pixel
4. Compute brightness and apparent number maps
5. Extract ROI-based statistics using segmentation masks
6. Export quantitative results to text file

In [4]:
# Import required libraries
from skimage import io
import glob
import os
import matplotlib.pyplot as plt
import numpy as np

## MAIN LOOP: PIXEL-WISE NUMBER & BRIGHTNESS CALCULATION

In [5]:
# Define input path pattern for batch processing
# Processes all .ptu files in the selected folder
path ='C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA*/*.ptu'

In [7]:
# Process each time series file
for file in glob.glob(path):
    # Extract base filename without extension
    filename = os.path.abspath(file).split(".")[0]
    
    # Load all polarization channels for this time series
    # Green channels (donor)
    img_green_s = io.imread( filename + '_green_s.tif')      # Green s-polarized
    img_green_p = io.imread( filename + '_green_p.tif')      # Green p-polarized
    
    # Red channels - prompt (immediate fluorescence)
    img_red_s_prompt = io.imread( filename + '_red_s_prompt.tif')    # Red s-polarized prompt
    img_red_p_prompt = io.imread( filename + '_red_p_prompt.tif')    # Red ps-polarized prompt
    
    img_red_s_delay = io.imread( filename + '_red_s_delay.tif')      # Red s-polarized delay
    img_red_p_delay = io.imread( filename + '_red_p_delay.tif')      # Red p-polarized delay
    
    # SELECT CHANNEL COMBINATION BASED ON SAMPLE TYPE:
    # Uncomment the appropriate line based on your experimental conditions
    
    #sum_img = img_green_s + img_green_p  # Donly samples
    #sum_img = img_red_s_delay + img_red_p_delay  # Acceptor only samples
    sum_img = img_green_s + img_green_p + img_red_s_prompt + img_red_p_prompt + img_red_s_delay + img_red_p_delay # FRET samples
    
    # Calculate temporal statistics for each pixel
    # mean_img: average intensity over time (proportional to concentration)
    mean_img = np.mean(sum_img, axis=0)
    io.imsave(filename + '_mean.tif', mean_img, check_contrast=False)
    
    # var_img: variance of intensity over time (measures fluctuations)
    var_img = np.var(sum_img, axis=0)
    io.imsave(filename + '_variance.tif', var_img, check_contrast=False)
    
    # Brightness = Variance/Mean (molecular brightness per particle)
    # Higher values indicate brighter individual molecules or larger complexes
    brightness = var_img / mean_img
    io.imsave(filename + '_brightness.tif', brightness, check_contrast=False)
    
    # Apparent Number = Mean²/Variance (number of particles per pixel)
    # Lower values may indicate aggregation or binding
    app_number = (mean_img * mean_img) /var_img
    io.imsave(filename + '_appNumber.tif', app_number, check_contrast=False)
    
    print('processing...' + filename)

C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:36: RuntimeWarning: invalid value encountered in divide
  brightness = var_img / mean_img
C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:41: RuntimeWarning: invalid value encountered in divide
  app_number = (mean_img * mean_img) /var_img


processing...C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64


C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:36: RuntimeWarning: invalid value encountered in divide
  brightness = var_img / mean_img
C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:41: RuntimeWarning: invalid value encountered in divide
  app_number = (mean_img * mean_img) /var_img


processing...C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66


C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:36: RuntimeWarning: invalid value encountered in divide
  brightness = var_img / mean_img
C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:41: RuntimeWarning: invalid value encountered in divide
  app_number = (mean_img * mean_img) /var_img


...

C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:36: RuntimeWarning: invalid value encountered in divide
  brightness = var_img / mean_img
C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:41: RuntimeWarning: invalid value encountered in divide
  app_number = (mean_img * mean_img) /var_img


processing...C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_72


C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:36: RuntimeWarning: invalid value encountered in divide
  brightness = var_img / mean_img
C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:41: RuntimeWarning: invalid value encountered in divide
  app_number = (mean_img * mean_img) /var_img


processing...C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_74
processing...C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_76


C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:36: RuntimeWarning: invalid value encountered in divide
  brightness = var_img / mean_img
C:\Users\Katherina Hemmen\AppData\Local\Temp\ipykernel_16480\1513932772.py:41: RuntimeWarning: invalid value encountered in divide
  app_number = (mean_img * mean_img) /var_img
